# Cross-Company Relative Position

Is a role paid fairly *relative to its peers at its own company*, and how does that relative position compare across companies?

Each target is placed against salaried roles in the **same seniority band at the same company** (median, IQR, percentile). Company pay level cancels out, so a technical writer at a high-paying company and one at a lower-paying company can be compared on the same footing.

Peer pools under 5 roles are shown but flagged as thin. Only roles that disclose salary count as peers, and disclosure follows pay-transparency law by location.

In [ ]:
# ── CONFIG ──────────────────────────────────────────────────────────
# (company, board, job_id) — technical-writer roles by default
TARGETS = [
    ("crusoe", "ashby", "2689707b-7314-4246-ac95-1e6466970ba3"),
    ("launchdarkly", "greenhouse", "7666736003"),
    ("redpandadata", "greenhouse", "4674278005"),
]

In [ ]:
import db as _db, pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from classify import add_usd_salary
from gold_analysis import build_relative_position, data_quality, latest_per_job

rows = []
_conn = _db.open_silver()
for company, board, job_id in TARGETS:
    df = pd.read_sql("SELECT * FROM jobs WHERE company=? AND board=?", _conn, params=[company, board])
    df["job_id"] = df["job_id"].astype(str)
    df = latest_per_job(df)
    print(f"{company}: {data_quality(df)}")
    add_usd_salary(df)
    df = df.dropna(subset=["mid_usd"])
    if job_id not in df["job_id"].values:
        print(f"  ⚠ {job_id} has no disclosed salary — skipped")
        continue
    rows.append({"company": company, **build_relative_position(df, job_id)})
_conn.close()

positions = pd.DataFrame(rows)

In [ ]:
# ── Table ───────────────────────────────────────────────────────────
if positions.empty:
    print("No targets with disclosed salary.")
else:
    table = positions.assign(
        target=positions["target_mid"].map("${:,.0f}".format),
        peers=positions.apply(lambda r: f"n={r['n']}{' ⚠ thin' if r['thin'] else ''}", axis=1),
        peer_median=positions["comp_median"].map("${:,.0f}".format),
        peer_iqr=positions.apply(lambda r: f"${r['q1']:,.0f}–${r['q3']:,.0f}", axis=1),
        gap=positions["gap_pct"].map("{:+.1f}%".format),
        pct=positions["percentile"].map("{:.0f}th".format),
    )[["company", "title", "seniority", "location", "target", "peers", "peer_median", "peer_iqr", "gap", "pct"]]
    display(table.style.hide(axis="index"))

In [ ]:
# ── Chart: target vs same-seniority peer IQR ────────────────────────
if not positions.empty:
    fig, ax = plt.subplots(figsize=(11, 1.2 + 0.9 * len(positions)))
    for i, r in positions.iterrows():
        ax.plot([r["q1"], r["q3"]], [i, i], color="#4a7cc9", linewidth=8, alpha=0.5, solid_capstyle="butt")
        ax.plot(r["comp_median"], i, "|", color="#1f3f7a", markersize=22, markeredgewidth=2)
        ax.plot(r["target_mid"], i, "o", color="#d62728", markersize=11, markeredgecolor="black", zorder=5)
        ax.text(max(r["q3"], r["target_mid"]) + 4000, i,
                f"{r['gap_pct']:+.0f}% vs median, n={r['n']}" + (" ⚠" if r["thin"] else ""), va="center", fontsize=9)
    ax.set_yticks(range(len(positions)))
    ax.set_yticklabels([f"{r['company'].title()}\n{r['title']}" for _, r in positions.iterrows()], fontsize=9)
    ax.invert_yaxis()
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x/1000:.0f}K"))
    ax.set_xlabel("Salary midpoint (USD, annualized)")
    ax.set_title("Target (red) vs same-seniority peers at the same company (band = IQR, tick = median)", fontsize=11)
    ax.grid(axis="x", alpha=0.3)
    plt.tight_layout()
    plt.show()